Denna fil tränar "Studenten".

Cell 1: Databeredning & Feature Engineering

extract: Vi skapar en "ledtråd" (ext_ip_flag) som hjälper modellen att förstå när en enhet pratar med fel person. Detta är direkt kopplat till din RAG-policy.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

# Ladda data
df = pd.read_json("network_audit_log.jsonl", lines=True)

# Feature Engineering: Gör om text/logik till siffror modellen förstår
def extract(row):
    m = row['metrics']
    srv_t = m["10.0.0.1"]["traffic"]
    iot_t = m["192.168.1.100"]["traffic"]
    # Viktiga flaggor baserade på vår RAG-policy
    is_external = 1 if "45." in str(m["192.168.1.100"]["notes"]) else 0
    return pd.Series([srv_t, iot_t, is_external])

X = df.apply(extract, axis=1)
X.columns = ['srv_traffic', 'iot_traffic', 'ext_ip_flag']
y = df['target']

print("Klassfördelning (ska ha både 0 och 1):")
print(y.value_counts())

Klassfördelning (ska ha både 0 och 1):
target
1    64
0    36
Name: count, dtype: int64


Cell 2: Robust träning med Stratify

stratify=y: Säkerställer att både tränings- och testdatan har samma andel attacker.

GridSearchCV: Testar olika hastigheter (learning_rate) för att hitta den mest pricksäkra modellen.

In [4]:
# Dela upp data med stratify=y eftersom vi nu har tillräckligt med attacker (tack vare 100 ticks)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Definiera modellen och sök efter bästa inställningar
param_grid = {'learning_rate': [0.01, 0.1], 'max_iter': [100, 200]}
grid = GridSearchCV(HistGradientBoostingClassifier(), param_grid, cv=3, scoring='f1')

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

print(f"Träning klar! Bästa F1-score: {grid.best_score_:.2%}")

Träning klar! Bästa F1-score: 77.87%


Cell 3: Resultat & Visualisering
Permutation_importance: Bevisar att din AI faktiskt lärt sig reglerna. Om ext_ip_flag är högst upp har din RAG-distillering fungerat perfekt!

classification_report: Ger dig de exakta siffrorna (Precision, Recall, F1) som du behöver för din rapport.